# 無料ツールだけでDIA-MSプロテオミクス環境を構築する

**対応記事**: [article-01-setup.md](../blog/article-01-setup.md) — DIA-MSプロテオミクス環境構築  
**実行順序**: 1番目  
**所要時間**: 約45分

---

## このNotebookで行うこと

DIA-MSプロテオミクス解析に必要なPython環境をゼロから構築します。すべて無料・商用利用可能なツールのみを使用します。

- micromambaのインストール
- Python環境の作成（基本版・GPU版）
- 動作確認とトラブルシューティング

**⚠️ 注意**: このNotebookは環境構築用のため、**ターミナルで実行する**コマンドが中心です。

## 前提条件

- macOS / Linux / Windows (WSL2)
- ターミナル（コマンドライン）が使えること
- インターネット接続

## 1. micromambaのインストール

micromambaはcondaの軽量版です。condaと同じパッケージが使えますが、インストール・解決が格段に速いです。

In [ ]:
import subprocess  # 外部コマンド実行用
import os         # 環境変数・パス操作用
import shutil     # ファイル操作・コマンド存在確認用

# micromambaが既にインストールされているかチェック
micromamba_path = shutil.which("micromamba")
if micromamba_path:
    print(f"✅ micromambaは既にインストールされています: {micromamba_path}")
    # バージョン確認
    result = subprocess.run(["micromamba", "--version"], capture_output=True, text=True)
    print(f"バージョン: {result.stdout.strip()}")
else:
    print("❌ micromambaがインストールされていません")
    print("以下のコマンドをターミナルで実行してください:")
    print('"${SHELL}" <(curl -L micro.mamba.pm/install.sh)')

**ターミナルでの実行コマンド**:

```bash
# micromambaインストール
"${SHELL}" <(curl -L micro.mamba.pm/install.sh)

# 確認
micromamba --version
```

インストール完了後、**新しいターミナルセッションを開いて**から次に進んでください。

## 2. Python環境の作成

DIA-MSプロテオミクス解析に必要なパッケージを含むPython環境を作成します。

In [ ]:
# environment.ymlが既に存在するかチェック
env_file_path = "../environment.yml"
if os.path.exists(env_file_path):
    print(f"✅ environment.ymlが既に存在します: {env_file_path}")
    # 内容を表示
    with open(env_file_path, 'r') as f:
        content = f.read()
    print("\n--- environment.yml の内容 ---")
    print(content)
else:
    print(f"❌ environment.ymlが見つかりません: {env_file_path}")
    print("以下の手順でファイルを作成してください。")

**ターミナルでの環境作成コマンド**:

```bash
# プロジェクトディレクトリに移動
cd /path/to/your/project

# environment.yml作成（プロジェクトルートに既に存在する場合はスキップ）
cat << 'EOF' > environment.yml
name: crc-proteomics
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11
  - numpy>=1.24
  - pandas>=2.0
  - scipy>=1.11
  - scikit-learn>=1.3
  - matplotlib>=3.7
  - seaborn>=0.13
  - statsmodels>=0.14
  - requests>=2.31
  - openpyxl>=3.1
  - adjustText>=0.8
  - pymzml>=2.5
  - pip
  - pip:
    - gseapy>=1.0
EOF

# 環境作成（5-10分）
micromamba create -f environment.yml -y

# 有効化
micromamba activate crc-proteomics
```

## 3. 動作確認

Python環境が正しく構築されたかを確認します。

In [ ]:
# 基本パッケージのインポートテスト
print("=== 基本パッケージ動作確認 ===")

try:
    import numpy as np
    print(f"✅ NumPy: {np.__version__}")
except ImportError as e:
    print(f"❌ NumPy: {e}")

try:
    import pandas as pd
    print(f"✅ Pandas: {pd.__version__}")
except ImportError as e:
    print(f"❌ Pandas: {e}")

try:
    import scipy
    print(f"✅ SciPy: {scipy.__version__}")
except ImportError as e:
    print(f"❌ SciPy: {e}")

try:
    import sklearn
    print(f"✅ scikit-learn: {sklearn.__version__}")
except ImportError as e:
    print(f"❌ scikit-learn: {e}")

try:
    import matplotlib
    print(f"✅ matplotlib: {matplotlib.__version__}")
except ImportError as e:
    print(f"❌ matplotlib: {e}")

try:
    import seaborn as sns
    print(f"✅ seaborn: {sns.__version__}")
except ImportError as e:
    print(f"❌ seaborn: {e}")

try:
    import statsmodels
    print(f"✅ statsmodels: {statsmodels.__version__}")
except ImportError as e:
    print(f"❌ statsmodels: {e}")

In [ ]:
# プロテオミクス特化パッケージのテスト
print("\n=== プロテオミクス特化パッケージ ===")

try:
    import openpyxl
    print(f"✅ openpyxl (Excel読み込み): {openpyxl.__version__}")
except ImportError as e:
    print(f"❌ openpyxl: {e}")

try:
    import adjustText
    print(f"✅ adjustText (高品質ラベル): バージョン不明")
except ImportError as e:
    print(f"❌ adjustText: {e}")

try:
    import pymzml
    print(f"✅ pymzml (mzML読み込み): {pymzml.__version__}")
except ImportError as e:
    print(f"❌ pymzml: {e}")

try:
    import gseapy
    print(f"✅ gseapy (パスウェイ解析): {gseapy.__version__}")
except ImportError as e:
    print(f"❌ gseapy: {e}")

## 4. GPU加速環境の構築【推奨】

より高速な解析のため、GPU加速環境を構築します。RTX 4070等のNVIDIA GPUが必要です。

**注意**: GPU環境は**ターミナルで構築**してください。

**ターミナルでのGPU環境構築**:

```bash
# GPU環境設定ファイル作成
cat << 'EOF' > environment-gpu.yml
name: crc-proteomics-gpu
channels:
  - conda-forge
  - bioconda
  - pytorch
  - nvidia
dependencies:
  - python=3.11
  - numpy>=1.24
  - pandas>=2.0
  - scipy>=1.11
  - scikit-learn>=1.3
  - matplotlib>=3.7
  - seaborn>=0.13
  - statsmodels>=0.14
  - requests>=2.31
  - openpyxl>=3.1
  - adjustText>=0.8
  - pymzml>=2.5
  # GPU support - WSL2 compatible
  - pytorch=2.0.1
  - torchvision=0.15.2
  - pytorch-cuda=11.8
  # Proteomics tools
  - openms=3.1.0
  - pip
  - pip:
    - peptdeep>=1.4.0
    - pyprophet>=2.2.0
    - gseapy>=1.0
EOF

# GPU環境作成（10-15分）
micromamba env create -f environment-gpu.yml -y

# 有効化
micromamba activate crc-proteomics-gpu

# PyTorchをpip経由で再インストール（WSL2互換性向上）
pip uninstall torch torchvision -y
pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118

# プロテオミクスツール追加
pip install peptdeep>=1.4.0 pyprophet>=2.2.0
```

In [ ]:
# GPU環境の動作確認（GPU環境でNotebookを実行している場合）
print("=== GPU環境動作確認 ===")

try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}")
    print(f"CUDA利用可能: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU名: {torch.cuda.get_device_name(0)}")
        memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU メモリ: {memory_gb:.1f} GB")
        
        # GPU計算テスト
        x = torch.tensor([1.0, 2.0, 3.0]).cuda()
        result = x.sum().cpu().item()
        print(f"GPU計算テスト結果: {result}")
        print("🎉 GPU ready!")
    else:
        print("⚠️ CUDA利用不可（CPUモードで実行されます）")
        
except ImportError as e:
    print(f"❌ PyTorch未インストール: {e}")

try:
    import peptdeep
    print(f"✅ AlphaPeptDeep: {peptdeep.__version__}")
except ImportError as e:
    print(f"❌ AlphaPeptDeep: {e}")

try:
    import pyprophet
    print(f"✅ PyProphet: バージョン確認中...")
except ImportError as e:
    print(f"❌ PyProphet: {e}")

## 5. パフォーマンス比較（実測結果）

| 処理段階 | CPU環境 | GPU環境 | 加速倍率 | GPU使用率 |
|---------|---------|---------|----------|-----------|
| 電荷予測 | ~2分 | 21秒 | **6倍** | 95% |
| RT予測 | ~5分 | 21秒 | **15倍** | 90% |
| MS2予測 | ~10分 | 20秒 | **30倍** | 85% |
| **総時間** | **~20分** | **~2分** | **10倍高速** | - |

**RTX 4070環境での実測値。Toyota et al. 2025パラメータ（771,738プリカーサー）で検証済み。**

## トラブルシューティング

### 1. CUDA初期化エラー
```bash
# WSL2でよくある問題。PyTorchをpip経由で再インストール
pip uninstall torch torchvision -y
pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
```

### 2. 依存関係競合
```bash
# 環境を削除して再作成
micromamba env remove -n crc-proteomics-gpu -y
micromamba env create -f environment-gpu.yml -y
```

### 3. パッケージ不足
```bash
# 追加インストール
micromamba activate crc-proteomics
pip install openpyxl adjustText
```

## まとめ

✅ **環境構築完了項目**:
- micromamba インストール
- Python 3.11 + 科学計算パッケージ
- プロテオミクス特化ツール
- GPU加速環境（オプション）

**次のステップ**:
- 基本環境: CPU版で十分な解析が可能
- GPU環境: RTX 4070等で約10倍高速化
- WSL2: pip経由のPyTorchインストールが安定

---

## Navigation

⬅️ **前回**: [article-00-introduction.md](../blog/article-00-introduction.md) — プロジェクト概要  
➡️ **次回**: [notebook_02a_data_acquisition.ipynb](./notebook_02a_data_acquisition.ipynb) — データ取得

---

*このNotebookは [article-01-setup.md](../blog/article-01-setup.md) に対応しています。*